In [3]:
import pandas as pd
import glob

# 1. Get the 8 daily price files
files = sorted(
    glob.glob("/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/*.csv")
)

print("Files found:", len(files))
print(files)

# 2. Read them safely
df_list = []

for f in files:
    temp = pd.read_csv(
        f,
        header=None,
        engine="python",
        on_bad_lines="skip"
    )
    df_list.append(temp)

Files found: 8
['/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-09.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-10.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-11.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-12.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-13.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-14.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-15.csv', '/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/price_2026-03-16.csv']


In [9]:
import pandas as pd
import glob

# 1. Read all files as raw text rows
files = sorted(
    glob.glob("/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/raw/price_extracted/*.csv")
)

all_rows = []

for f in files:
    with open(f, "r", errors="ignore") as file:
        for line in file:
            row = line.strip().split(",")
            all_rows.append(row)

print("Total rows loaded:", len(all_rows))

# 2. Convert to DataFrame
price_raw = pd.DataFrame(all_rows)

print("Raw shape:", price_raw.shape)
print(price_raw.head(10))

Total rows loaded: 23072
Raw shape: (23072, 138)
  0           1       2     3                      4           5         6    \
0   C  NEMP.WORLD  PRICES  AEMO                 PUBLIC  2026/03/09  04:05:02   
1   I     DREGION             2         SETTLEMENTDATE       RUNNO  REGIONID   
2   D     DREGION             2  "2026/03/08 18:25:00"           1      NSW1   
3   D     DREGION             2  "2026/03/08 18:25:00"           1      QLD1   
4   D     DREGION             2  "2026/03/08 18:25:00"           1       SA1   
5   D     DREGION             2  "2026/03/08 18:25:00"           1      TAS1   
6   D     DREGION             2  "2026/03/08 18:25:00"           1      VIC1   
7   D     DREGION             2  "2026/03/08 16:45:00"           1      NSW1   
8   D     DREGION             2  "2026/03/08 16:45:00"           1      QLD1   
9   D     DREGION             2  "2026/03/08 16:45:00"           1       SA1   

                7          8                 9    ...   128   129   13

In [10]:
price_raw = pd.DataFrame(all_rows)

price_raw.head(10)

,0,1,2,3,4,5,6,7,8,9,...,128,129,130,131,132,133,134,135,136,137
0,C,NEMP.WORLD,PRICES,AEMO,PUBLIC,2026/03/09,04:05:02,0000000506989736,,0000000506989730,...,None,None,None,None,None,None,None,None,None,None
1,I,DREGION,,2,SETTLEMENTDATE,RUNNO,REGIONID,INTERVENTION,RRP,EEP,...,None,None,None,None,None,None,None,None,None,None
2,D,DREGION,,2,"""2026/03/08 18:25:00""",1,NSW1,0,88.95495,0,...,None,None,None,None,None,None,None,None,None,None
3,D,DREGION,,2,"""2026/03/08 18:25:00""",1,QLD1,0,93.95,0,...,None,None,None,None,None,None,None,None,None,None
4,D,DREGION,,2,"""2026/03/08 18:25:00""",1,SA1,0,61.4195,0,...,None,None,None,None,None,None,None,None,None,None
5,D,DREGION,,2,"""2026/03/08 18:25:00""",1,TAS1,0,106.60955,0,...,None,None,None,None,None,None,None,None,None,None
6,D,DREGION,,2,"""2026/03/08 18:25:00""",1,VIC1,0,55.96,0,...,None,None,None,None,None,None,None,None,None,None
7,D,DREGION,,2,"""2026/03/08 16:45:00""",1,NSW1,0,99,0,...,None,None,None,None,None,None,None,None,None,None
8,D,DREGION,,2,"""2026/03/08 16:45:00""",1,QLD1,0,102.1,0,...,None,None,None,None,None,None,None,None,None,None
9,D,DREGION,,2,"""2026/03/08 16:45:00""",1,SA1,0,-41.97,0,...,None,None,None,None,None,None,None,None,None,None


In [11]:
# 3. Keep only actual data rows
price_df = price_raw[price_raw[0] == "D"].copy()

# 4. Keep only required columns
price_df = price_df[[4, 6, 8]].copy()
price_df.columns = ["SETTLEMENTDATE", "REGIONID", "RRP"]

# 5. Clean values
price_df["SETTLEMENTDATE"] = (
    price_df["SETTLEMENTDATE"]
    .astype(str)
    .str.replace('"', '', regex=False)
)

price_df["SETTLEMENTDATE"] = pd.to_datetime(
    price_df["SETTLEMENTDATE"],
    errors="coerce"
)

In [12]:
price_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 23040 entries, 2 to 23070
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   SETTLEMENTDATE  23040 non-null  datetime64[ns]
 1   REGIONID        23040 non-null  object        
 2   RRP             23040 non-null  object        
dtypes: datetime64[ns](1), object(2)
memory usage: 720.0+ KB


In [13]:

price_df["RRP"] = pd.to_numeric(price_df["RRP"], errors="coerce")

# 6. Filter VIC and required dates
price_df = price_df[price_df["REGIONID"] == "VIC1"].copy()

price_df = price_df[
    (price_df["SETTLEMENTDATE"] >= "2026-03-09") &
    (price_df["SETTLEMENTDATE"] < "2026-03-17")
].copy()

# 7. Remove duplicates and sort
price_df = price_df.drop_duplicates(subset=["SETTLEMENTDATE", "REGIONID"])
price_df = price_df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

# 8. Final checks
print("Clean shape:", price_df.shape)
print(price_df.head())
print(price_df.tail())
print(price_df.isna().sum())

Clean shape: (2065, 3)
       SETTLEMENTDATE REGIONID       RRP
0 2026-03-09 00:00:00     VIC1  60.22447
1 2026-03-09 00:05:00     VIC1  59.78197
2 2026-03-09 00:10:00     VIC1  72.69000
3 2026-03-09 00:15:00     VIC1  60.50989
4 2026-03-09 00:20:00     VIC1  55.96000
          SETTLEMENTDATE REGIONID       RRP
2060 2026-03-16 03:40:00     VIC1  60.49741
2061 2026-03-16 03:45:00     VIC1  60.14733
2062 2026-03-16 03:50:00     VIC1  57.88521
2063 2026-03-16 03:55:00     VIC1  66.93910
2064 2026-03-16 04:00:00     VIC1  65.49051
SETTLEMENTDATE    0
REGIONID          0
RRP               0
dtype: int64


In [14]:
price_df.to_csv(
    "/Users/vivekarya/Documents/GitHub/bess_trading_assessment/data/processed/price_clean.csv",
    index=False
)

print("Saved: price_clean.csv")

Saved: price_clean.csv


In [15]:
print(price_df["REGIONID"].unique())
print(price_df["SETTLEMENTDATE"].min(), price_df["SETTLEMENTDATE"].max())
print(price_df["RRP"].describe())

['VIC1']
2026-03-09 00:00:00 2026-03-16 04:00:00
count    2065.000000
mean       20.863206
std        38.709533
min       -60.000000
25%        -5.421550
50%         8.950000
75%        54.690000
max       136.164310
Name: RRP, dtype: float64
